Grupo 4 - 19/05/2026 - IA - Trabajo IA - Extracción de Embeddings

# Notebook 3: Extracción de Embeddings Faciales

En el Notebook 2 detectamos y recortamos todas las caras del dataset del Gobierno de La Rioja, normalizándolas a 160×160 px. En este notebook usaremos estos recortes para generar embeddings faciales. En este notebook buscaremos comparar tres modelos de reconocimiento facial(VGGFace, OpenFace y FaceNet) y elegir el mejor de estos y finalmente generaremos los embeddings para todos los recortes del dataset.

## 0. Carga de librerías

Cargamos las librerías necesarias para la deteccion.

In [1]:
%%capture
!pip install deepface tf-keras

In [2]:
import os
import cv2
import pickle
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from deepface import DeepFace
%matplotlib inline

26-05-19 09:05:20 - Directory /root/.deepface has been created
26-05-19 09:05:20 - Directory /root/.deepface/weights has been created


## 1. Carga del Dataset de Caras

Descargamos el ZIP con los recortes generados en el Notebook 2 desde el release de GitHub.

In [3]:
%%capture
!wget FACE_CROPS_URL -O carasRecortadas.zip
!unzip carasRecortadas.zip -d carasRecortadas

In [4]:
datasetPath = Path('/content/carasRecortadas')

caras = sorted(datasetPath.glob('*.jpg'))

print(f"Total de imágenes: {len(caras)}")

Total de imágenes: 38299


## 2. ¿Qué es un Embedding Facial?

Un embedding facial es un vector numérico de dimensión fija que representa la identidad de una cara. El modelo transforma la imagen de entrada en ese vector de forma que:

- Dos imágenes de la misma persona producen vectores próximos entre sí.
- Dos imágenes de personas distintas producen vectores lejanos.

Es esta propiedad es la que nos permite construir un motor de búsqueda facial.

## 3. Comparativa de Modelos

Comparamos tres modelos de reconocimiento facial disponibles en DeepFace:

| Modelo | Dimensión embedding | Arquitectura |
|---|---|---|
| **VGG-Face** | 4096 | VGGNet |
| **OpenFace** | 128 | SqueezeNet |
| **FaceNet** | 128 | Inception |

Los evaluamos sobre una muestra de 1000 recortes midiendo tiempo y dimensión del embedding resultante.

In [5]:
def rendimiento_modelo(modelo, muestra):
    t0 = time.time()
    embeddings = [
        DeepFace.represent(img_path=str(ruta), model_name=modelo, enforce_detection=False)[0]['embedding']
        for ruta in muestra
    ]
    tf = time.time() - t0
    print(f"{modelo:<15} dim={len(embeddings[0]):<6} {tf:.2f}s")
    return embeddings

In [6]:
random.seed(42)
muestra = random.sample(caras, 1000)

### 3.1 Modelo 1 : VGG-Face

In [7]:
embeddings_vgg = rendimiento_modelo('VGG-Face', muestra)

26-05-19 09:05:40 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /root/.deepface/weights/vgg_face_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /root/.deepface/weights/vgg_face_weights.h5
100%|██████████| 580M/580M [00:06<00:00, 90.8MB/s]


VGG-Face        dim=4096   82.67s


### 3.2 Modelo 2 : OpenFace

In [8]:
embeddings_openface = rendimiento_modelo('OpenFace', muestra)

26-05-19 09:07:02 - 🔗 openface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5 to /root/.deepface/weights/openface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5
To: /root/.deepface/weights/openface_weights.h5
100%|██████████| 15.3M/15.3M [00:00<00:00, 118MB/s] 


OpenFace        dim=128    136.03s


### 3.3 Modelo 3 : Facenet

In [9]:
embeddings_facenet = rendimiento_modelo('Facenet', muestra)

26-05-19 09:09:19 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5
100%|██████████| 92.2M/92.2M [00:00<00:00, 128MB/s]


Facenet         dim=128    280.24s


### 3.4 Resultados

| Modelo    | Dimensión | Tiempo  | imgs/s |
|-----------|----------:|-------------------:|-------:|
| VGG-Face  | 4096      | 82.67s             | 12.1   |
| OpenFace  | 128       | 136.03s            | 7.4    |
| Facenet   | 128       | 280.24s            | 3.6    |

El resultado más llamativo es que VGG-Face es el más rápido pese a producir embeddings más grandes. Buscandolo, esto se explica por que es el que lleva mas año de soporte y el que esta mejor optimizado para la GPU de google colab

Para el producto final, la elección del modelo depende de dos cosas:
- la calidad de los embeddings como representación de identidad
- el coste de búsqueda en el VP-Tree del Notebook 4.

En cuanto a la búsqueda, el VP-Tree opera en O(log n) pero ese coste crece con la dimensión del espacio, ya que calcular distancias en 4096d es ~32× más caro por consulta que en 128d. Con 38.299 embeddings indexados, la diferencia será apreciableen cada búsqueda. OpenFace y Facenet parten por tanto con ventaja estructural para el motor de búsqueda.

Entre OpenFace y Facenet, ambos producen vectores de 128 dimensiones, pero Facenet está entrenado sobre datasets mucho más grandes (VGGFace2, MS-Celeb) y tiene consistentemente mejor separabilidad entre identidades en benchmarks estándar comoLFW. Su mayor coste de extracción (280s vs 136s) es un pago único al generar el índice, no un coste recurrente en cada consulta.

Por tanto, **Facenet es la elección para el Notebook 4**: ofrece la mejor calidad de embedding para reconocimiento facial, sus vectores de 128d mantienen el VP-Tree eficiente en consulta, y el mayor tiempo de indexación es irrelevante al ejecutarlo una sola vez con GPU local.

## 4. Generación de Embeddings sobre el Dataset Completo

Generamos un embedding FaceNet de 128 dimensiones para cada recorte del dataset. El resultado se almacena en un diccionario `{nombre_fichero: vector}` que guardamos en `embeddings.pkl`.

Usamos `enforce_detection=False` porque los recortes ya están normalizados desde el Notebook 2 — no necesitamos que DeepFace vuelva a detectar la cara dentro del recorte.

Tal y como ya lo hicimos para el notebook 1 y 2, para manejar todo el dataset lo ejecutaremos localmente

In [ ]:
import time
import pickle
import numpy as np
from pathlib import Path
from deepface import DeepFace
from collections import defaultdict

In [ ]:
carasPath = Path('carasRecortadas')
caras = [p for p in carasPath.rglob('*.jpg')]
print(f"Caras encontradas: {len(caras)}")

In [ ]:
# Forzamos la carga del modelo antes del bucle para que el tiempo
# de descarga no contamine la medición, igual que en NB2 con dlib CNN
DeepFace.build_model('Facenet')
print("Modelo Facenet cargado en GPU.")

In [ ]:
embeddings = {}
contadores = defaultdict(int)

print(f"Procesando {len(caras)} caras...")
t_inicio = time.time()

for idx, ruta in enumerate(caras):
    try:
        emb = DeepFace.represent(
            img_path=str(ruta),
            model_name='Facenet',
            enforce_detection=False
        )[0]['embedding']
        embeddings[ruta.name] = emb
        contadores['ok'] += 1
    except Exception:
        contadores['error'] += 1

    if idx % 1000 == 0:
        print(f"  {idx}/{len(caras)} — {contadores['ok']} embeddings...")

tf = time.time() - t_inicio
print(f"\nCompletado en {tf:.1f}s ({tf/60:.1f} min)")
print(f"  Embeddings generados : {contadores['ok']}")
print(f"  Errores              : {contadores['error']}")

In [ ]:
OUT = Path('embeddings_facenet.pkl')
with open(OUT, 'wb') as f:
    pickle.dump(embeddings, f)
print(f"Guardado en {OUT}  ({OUT.stat().st_size / 1024 / 1024:.1f} MB)")